# Kaggle Playground S6E10 - Predicting Airline Satisfaction

## Section 1 - EDA

### Predictions 

1. Satisfaction roughly split 50-50 between True and False
2. Important columns for predictions are: Age, Flight Distance, Inflight wifi service, Ease of Online booking, Food and drink, Seat comfort, On-board service,Leg room service, Departure Delay in Minutes, Arrival Delay in Minutes, Customer Type (loyal/disloyal)
3. Train and test might not look alike - potential for overfitting

### Potential issues to investigate

- Does inflight entertainment correlate with flight distance? 
- Nulls in departure/arrival delays might mean data might not have been captured at all - cancelled, diverted, arrival time not logged

### Load data

In [ ]:
# all imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')  

In [ ]:
# load raw data and store in dataframes
data = pd.read_csv('../data/raw/train.csv')
test_data = pd.read_csv('../data/raw/test.csv')

In [ ]:
data.shape # approx 700k rows, 23 columns

(699635, 23)

In [ ]:
test_data.shape # approx 300k rows, 22 columns

(299844, 22)

In [ ]:
# test_data is missing satisfaction, the target variable
set(data.columns) - set(test_data.columns)

# data.columns.difference(test_data.columns) # this gets the same result

{'satisfaction'}

In [12]:
data.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


In [13]:
test_data.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class
0,699635,25,1197,4,4,4,2,3,4,3,...,5,2,1,3,0,0.0,Female,disloyal Customer,Business travel,Eco
1,699636,36,829,5,5,5,5,5,4,5,...,2,4,3,5,0,0.0,Male,Loyal Customer,Business travel,Eco
2,699637,45,472,4,5,5,5,4,4,4,...,3,1,3,4,0,0.0,Male,Loyal Customer,Business travel,Eco
3,699638,45,762,2,2,2,5,3,2,3,...,4,4,4,3,0,0.0,Female,disloyal Customer,Business travel,Business
4,699639,34,594,1,2,1,1,2,1,2,...,4,2,1,2,0,0.0,Male,Loyal Customer,Personal Travel,Eco Plus


In [22]:
# describe training data
data.describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
id,699635.0,349817.0,201967.4,0.0,174908.5,349817.0,524725.5,699634.0
Age,699635.0,39.0,13.8,7.0,27.0,40.0,50.0,85.0
Flight Distance,699635.0,1353.8,954.4,67.0,631.0,954.0,1814.0,4983.0
Inflight wifi service,699635.0,2.8,1.3,0.0,2.0,3.0,4.0,5.0
Departure/Arrival time convenient,699635.0,3.2,1.5,0.0,2.0,3.0,4.0,5.0
Ease of Online booking,699635.0,2.8,1.3,0.0,2.0,3.0,4.0,5.0
Gate location,699635.0,3.0,1.3,0.0,2.0,3.0,4.0,5.0
Food and drink,699635.0,3.3,1.3,0.0,2.0,3.0,4.0,5.0
Online boarding,699635.0,3.4,1.3,0.0,2.0,4.0,4.0,5.0
Seat comfort,699635.0,3.6,1.3,0.0,3.0,4.0,5.0,5.0


In [24]:
# arrival delay is not hitting max rows - checking for missing values across other variables
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departure/Arrival time convenient  699635 non-null  int64  
 5   Ease of Online booking             699635 non-null  int64  
 6   Gate location                      699635 non-null  int64  
 7   Food and drink                     699635 non-null  int64  
 8   Online boarding                    699635 non-null  int64  
 9   Seat comfort                       699635 non-null  int64  
 10  Inflight entertainment             699635 non-null  int64  
 11  On-board service                   699635 non-null

In [ ]:
# checking variables with at least 1 NA
na = data.isna().sum()
na[na > 0] 

# 292 missing Arrival Delay in Minutes values in training data 

Arrival Delay in Minutes    292
dtype: int64

In [33]:
# what fraction of rows are missing Arrival Delay in Minutes?
pct = round(100 * data["Arrival Delay in Minutes"].isna().mean(), 2)
print(f"{pct:.2f}%")

0.04%


In [ ]:
# checking if missingness is random or if it relates to anything: 

data.groupby(data["Arrival Delay in Minutes"].isna())["satisfaction"].mean()

# satisfaction significantly lower given absence of Arrival Delay - need to identify this cause

Arrival Delay in Minutes
False    0.443606
True     0.363014
Name: satisfaction, dtype: float64

In [ ]:
# how many rows are in the missing group? getting a N for this case
data.groupby(data["Arrival Delay in Minutes"].isna())["satisfaction"].agg(["mean", "count"])

"""
292 True -> standard error rate is around 0.03, gap of 0.077 is around 2.6 standard errors
Under 5% significance level, could possibly be a real difference
Later to-do: run a proper test on these real counts to confirm, e.g. scipy.stats.chi2_contingency on 2x2 table of missing/not missing vs satisfied/not satisfied

Handling data: tree models eg LightGBM, XGBoost, leave NaNs in, as they can learn a seperate direction for missing values
Models that need filled values, feature engineer a 0/1 column such as arriving_delay_missing before you fill NaNs
"""

,mean,count
Arrival Delay in Minutes,,
False,0.443606,699343
True,0.363014,292


In [ ]:
# also checking missing variables in test data:
na = test_data.isna().sum()
na[na > 0]

# 130 missing Arrival Delay in Minutes values in test data

Arrival Delay in Minutes    130
dtype: int64

In [34]:
# for test data, what fraction of rows are missing Arrival Delay in Minutes?
pct_test = round(100 * test_data["Arrival Delay in Minutes"].isna().mean(), 2)
print(f"{pct_test:.2f}%")

0.04%
